# Laboratorio 4

In [1]:

import rasterio
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
import openeo

### Ejercicio 1 

Establezca una conexión con la api de sentinel 2, puede usar para eso el módulo openeo.


In [2]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=ZALM-RCSZ 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.


### Ejercicio 2

Descarga los .tif de cada uno de los lagos usando para eso las coordenadas o el geojson provisto


In [5]:
import json

with open("./archive/Lago_Amatitlan.geojson", "r", encoding="utf-8") as archivo:
    amatitlan = json.load(archivo)

with open("./archive/Lago_Atitlan.geojson", "r", encoding="utf-8") as archivo:
    atitlan = json.load(archivo)


print("Amatitlan, ", amatitlan)
print("Atitlan, ", atitlan)

Amatitlan,  {'type': 'FeatureCollection', 'name': 'Lago_Amatitlan', 'crs': {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}}, 'features': [{'type': 'Feature', 'properties': {'name': 'Lago Amatitlán'}, 'geometry': {'type': 'Polygon', 'coordinates': [[[-90.512924, 14.412347], [-90.512924, 14.493799], [-90.638065, 14.493799], [-90.638065, 14.412347], [-90.512924, 14.412347]]]}}]}
Atitlan,  {'type': 'FeatureCollection', 'name': 'Lago_Atitlan', 'crs': {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}}, 'features': [{'type': 'Feature', 'properties': {'name': 'Lago Atitlán'}, 'geometry': {'type': 'Polygon', 'coordinates': [[[-91.07151, 14.5948], [-91.07151, 14.750979], [-91.326256, 14.750979], [-91.326256, 14.5948], [-91.07151, 14.5948]]]}}]}


In [11]:
geometry_atl = atitlan["features"][0]["geometry"]["coordinates"][0]

longitudes_atl = [punto[0] for punto in geometry_atl]
latitudes_atl = [punto[1] for punto in geometry_atl]

crd_atl = {
  "west": min(longitudes_atl),
    "east": max(longitudes_atl),
    "south": min(latitudes_atl),
    "north": max(latitudes_atl)
}

  

Cargando los datos en cubes

In [ ]:

atitlan_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=crd_atl,
    temporal_extent=["2025-01-01", "2025-08-01"],
    bands=["B02","B03","B04", "B08"]
)

In [ ]:
# Carga todo el rango
atitlan_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=crd_atl,
    temporal_extent=["2025-03-01", "2025-08-31"],
    bands=["B02","B03","B04","B08"]
)

# Luego filtras para quedarte con 2 imágenes por mes:
def select_two_per_month(cube):
    selected = []
    times = cube.time.to_index()  # o la forma que tenga para fechas
    for month in range(3, 9):
        month_dates = [t for t in times if t.month == month]
        # Escoge 2 fechas arbitrarias, por ejemplo las primeras 2 disponibles
        selected_dates = month_dates[:2]
        selected.extend(selected_dates)
    # Filtra el cube con estas fechas
    return cube.sel(time=selected)
print(atitlan_cube.dims)
print(atitlan_cube.coords)

atitlan_cube_filtered = select_two_per_month(atitlan_cube)


AttributeError: 'DataCube' object has no attribute 'dims'

In [ ]:
result_graph = atitlan_cube_filtered.save_result(format="GTIFF")
job = connection.create_job(result_graph)

# Iniciar y esperar que termine
job.start_and_wait()

# Obtener resultados y descargarlos a carpeta
results = job.get_results()
results.download_files("./data/GIS/")

print("Descarga finalizada.")

# Verificar archivos descargados
import os

folder = "./data/GIS/"
files = os.listdir(folder)
for f in files:
    size = os.path.getsize(os.path.join(folder, f))
    print(f"{f} - {size} bytes")

# Imprimir logs del job por si hubo errores
print("\nLogs del job:")
print(job.get_log())


0:00:00 Job 'j-2508072241104717b7906a97362ba7f9': send 'start'
0:00:13 Job 'j-2508072241104717b7906a97362ba7f9': created (progress 0%)
0:00:18 Job 'j-2508072241104717b7906a97362ba7f9': created (progress 0%)
0:00:24 Job 'j-2508072241104717b7906a97362ba7f9': created (progress 0%)
0:00:32 Job 'j-2508072241104717b7906a97362ba7f9': created (progress 0%)
0:00:43 Job 'j-2508072241104717b7906a97362ba7f9': created (progress 0%)
0:00:55 Job 'j-2508072241104717b7906a97362ba7f9': created (progress 0%)
0:01:11 Job 'j-2508072241104717b7906a97362ba7f9': running (progress N/A)
0:01:30 Job 'j-2508072241104717b7906a97362ba7f9': running (progress N/A)
0:01:54 Job 'j-2508072241104717b7906a97362ba7f9': running (progress N/A)
0:02:24 Job 'j-2508072241104717b7906a97362ba7f9': running (progress N/A)
0:03:02 Job 'j-2508072241104717b7906a97362ba7f9': running (progress N/A)
0:03:48 Job 'j-2508072241104717b7906a97362ba7f9': running (progress N/A)
0:04:47 Job 'j-2508072241104717b7906a97362ba7f9': running (progress

KeyboardInterrupt: 